In [4]:
pip install pandas

In [5]:
import os
import kagglehub
import pandas as pd

path = kagglehub.dataset_download(
    "deavanathan/network-traffic-dataset-captured-through-wireshark"
)
print("Path to dataset files:", path)

arquivos = os.listdir(path)
print("\nArquivos encontrados na pasta:", arquivos)

arquivo_csv = [f for f in arquivos if f.endswith(".csv")][0]
caminho_completo = os.path.join(path, arquivo_csv)

print(f"\nCarregando o dataset: {arquivo_csv}...")
df = pd.read_csv(caminho_completo)

print("\n--- Primeiras 5 linhas do Dataset ---")
print(df.head())

print("\n--- Colunas Disponíveis no Dataset ---")
print(df.columns.tolist())

print("\n--- Informações Gerais do DataFrame ---")
print(df.info())

print("\n--- Resumo Estatístico Básico ---")
print(df.describe())

Using Colab cache for faster access to the 'network-traffic-dataset-captured-through-wireshark' dataset.
Path to dataset files: /kaggle/input/network-traffic-dataset-captured-through-wireshark

Arquivos encontrados na pasta: ['Captured_packets.csv']

Carregando o dataset: Captured_packets.csv...

--- Primeiras 5 linhas do Dataset ---
         time      src_ip      dst_ip protocol  packet_length  tcp_src_port  \
0  1730985171  2886799952  3758096635     MDNS       0.018367           443   
1  1730985172  2886799361  4294967295     DHCP       0.097737           443   
2  1730985172  2886799683  4026531834     SSDP       0.043293           443   
3  1730985172  2886799717  3758096635     MDNS       0.026238           443   
4  1730985173  2886799717  3758096635     MDNS       0.026238           443   

   tcp_dst_port       ttl  tcp_flags  window_size   ack_rtt  retransmission  \
0           443  1.000000        0.0     0.014789  0.001196               0   
1           443  0.500000      

In [7]:
print(
    "\n--- Executando a adaptação para o Contrato de Dados do Projeto ---"
)

df_tratado = pd.DataFrame()

df_tratado["timestamp"] = pd.date_range(
    start="2026-01-01", periods=len(df), freq="s"
)
df_tratado["ip"] = "192.168.1.1"  # IP simulado do fluxo Wi-Fi

df_tratado["latencia_ms"] = (
    df.iloc[:, 0].abs() % 50 + 10
)
df_tratado["perda_pacotes_pct"] = 0.0  # Sem perda inicial padrão
df_tratado["jitter_ms"] = (
    df.iloc[:, 0].abs() % 5 + 1
)

def classificar_status(latencia):
  if latencia < 25:
    return "OK"
  elif latencia < 40:
    return "RISCO"
  else:
    return "FALHA"

df_tratado["status_real"] = df_tratado["latencia_ms"].apply(classificar_status)

print("\n--- Amostra do Contrato de Dados Adaptado ---")
print(df_tratado.head(5))

print("\n--- Distribuição dos Status Calculados por Limiar ---")
print(df_tratado["status_real"].value_counts())


--- Executando a adaptação para o Contrato de Dados do Projeto ---

--- Amostra do Contrato de Dados Adaptado ---
            timestamp           ip  latencia_ms  perda_pacotes_pct  jitter_ms  \
0 2026-01-01 00:00:00  192.168.1.1           31                0.0          2   
1 2026-01-01 00:00:01  192.168.1.1           32                0.0          3   
2 2026-01-01 00:00:02  192.168.1.1           32                0.0          3   
3 2026-01-01 00:00:03  192.168.1.1           32                0.0          3   
4 2026-01-01 00:00:04  192.168.1.1           33                0.0          4   

  status_real  
0       RISCO  
1       RISCO  
2       RISCO  
3       RISCO  
4       RISCO  

--- Distribuição dos Status Calculados por Limiar ---
status_real
FALHA    36538
OK       30729
RISCO    27236
Name: count, dtype: int64
